# 01 — Is the bottleneck compression or comparability?

Controlled endpoint-only ablation. Every row uses the same corpus, student,
objective, schedule and three training seeds. Random subspaces additionally use
three independent draws. The output is Appendix Table A1; learned maps have no
fixed teacher target, so intrinsic-retention columns are intentionally undefined.

In [ ]:
# 1. Settings
from pathlib import Path

REPO_URL = "https://github.com/duncan-nguyen/embedding-kd.git"
AUTO_PULL_REPO, INSTALL_REQUIREMENTS = True, True
PAIR = "qwen3_0.6b_to_minilm_h384"
TRAIN_DATA_REL = Path("data/train_set/train_100k.csv")
RUN_NAME = f"analysis_interface_{PAIR}_v2"
SEEDS, RANDOM_DRAWS = [42], [0]
BATCH_SIZE, EPOCHS, LR = 128, 5, 7e-5
STUDENT_DIM, GEOMETRY_ROWS = 384, 1024
LEARNED_LR_SCALE = 1.0
EXECUTE, STOP_ON_ERROR, REQUIRE_COMPLETE = True, True, True
CUDA_VISIBLE_DEVICES = "0"

In [3]:
# 2. Repo, dependencies, GPU và dữ liệu
import subprocess, sys

cwd = Path.cwd().resolve()
PROJECT_DIR = next((p for p in (cwd, cwd.parent) if (p / "main.py").is_file()), None)
if PROJECT_DIR is None:
    clone_parent = Path("/content") if Path("/content").is_dir() else cwd
    PROJECT_DIR = clone_parent / "embedding-kd"
    if not PROJECT_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
    assert (PROJECT_DIR / "main.py").is_file(), f"Repo không hợp lệ: {PROJECT_DIR}"

tracked = subprocess.run(
    ["git", "-C", str(PROJECT_DIR), "status", "--porcelain", "--untracked-files=no"],
    check=True, capture_output=True, text=True,
).stdout.strip()
if AUTO_PULL_REPO and not tracked:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)
elif AUTO_PULL_REPO:
    print("[git] Bỏ qua pull vì repo có tracked changes.")
if INSTALL_REQUIREMENTS:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", str(PROJECT_DIR / "requirements.txt")],
        check=True,
    )

git_head = subprocess.run(
    ["git", "-C", str(PROJECT_DIR), "rev-parse", "--short", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()
sys.path[:0] = [str(PROJECT_DIR), str(PROJECT_DIR / "notebooks")]
TRAIN_DATA = PROJECT_DIR / TRAIN_DATA_REL
assert TRAIN_DATA.is_file(), f"Thiếu training data: {TRAIN_DATA}"
if EXECUTE:
    import torch
    assert torch.cuda.is_available(), "Hãy bật GPU runtime trước khi train."
print(f"Repo: {PROJECT_DIR} @ {git_head}")
print(f"Training data: {TRAIN_DATA}")

Repo: /content/embedding-kd @ bc022f3
Training data: /content/embedding-kd/data/train_set/train_100k.csv


In [4]:
# 3. Plan: one job per (interface, draw, seed)
import shlex
from _analysis_common import PAIRS, collect_jobs, geoode_command, run_jobs

pair, cache_dir = PAIRS[PAIR], PROJECT_DIR / "runs" / "teacher_cache"
run_root = PROJECT_DIR / "runs" / RUN_NAME
run_root.mkdir(parents=True, exist_ok=True)
specs = [
    ("learned_t2s", [None], ["--projection_type", "learned_t2s", "--no-gauge_align", "--learned_projector_lr_scale", LEARNED_LR_SCALE]),
    ("learned_s2t", [None], ["--projection_type", "learned_s2t", "--no-gauge_align", "--learned_projector_lr_scale", LEARNED_LR_SCALE]),
    ("random_only", RANDOM_DRAWS, ["--projection_type", "random", "--no-gauge_align"]),
    ("random_proc", RANDOM_DRAWS, ["--projection_type", "random", "--gauge_align", "--gauge_rotation", "procrustes"]),
    ("pca_only", [None], ["--projection_type", "pca", "--no-gauge_align"]),
    ("pca_proc", [None], ["--projection_type", "pca", "--gauge_align", "--gauge_rotation", "procrustes"]),
]
jobs = []
for arm, draws, arm_args in specs:
    for draw in draws:
        for seed in SEEDS:
            cell = arm if draw is None else f"{arm}__d{draw}"
            extra = ["--lambda_end", 1, "--lambda_ctr", 0, "--lambda_topo", 0,
                     "--gauge_refit_every", 1 if arm.endswith("proc") else 0,
                     "--no_eval_retrieval", *arm_args]
            if draw is not None:
                extra += ["--projection_seed", draw]
            run_dir = run_root / cell / f"seed_{seed}"
            jobs.append({
                "name": f"{cell}/seed_{seed}", "arm": arm, "draw": draw,
                "seed": seed, "run_dir": run_dir,
                "command": geoode_command(
                    PROJECT_DIR, pair=pair, train_data=TRAIN_DATA,
                    cache_dir=cache_dir, run_dir=run_dir, seed=seed,
                    batch_size=BATCH_SIZE, epochs=EPOCHS, learning_rate=LR,
                    extra=extra,
                ),
            })
print(f"Plan: {len(jobs)} jobs -> {run_root}")
for job in jobs:
    print(shlex.join(job["command"]))

Plan: 6 jobs -> /content/embedding-kd/runs/analysis_interface_qwen3_0.6b_to_minilm_h384_v2
/usr/bin/python3 /content/embedding-kd/main.py --method geoode --train_data /content/embedding-kd/data/train_set/train_100k.csv --student_model nreimers/MiniLMv2-L6-H384-distilled-from-BERT-Base --teacher_model Qwen/Qwen3-Embedding-0.6B --teacher_pooling last_token --batch_size 128 --epochs 5 --save_every 5 --lr 7e-05 --max_length 256 --seed 42 --num_workers 2 --eval_every 0 --pair_threshold_source validation --no-evaluate_test_each_epoch --cache_dir /content/embedding-kd/runs/teacher_cache --save_dir /content/embedding-kd/runs/analysis_interface_qwen3_0.6b_to_minilm_h384_v2/learned_t2s/seed_42 --no_wandb --student_pooling cls --lambda_end 1 --lambda_ctr 0 --lambda_topo 0 --gauge_refit_every 0 --no_eval_retrieval --projection_type learned_t2s --no-gauge_align --learned_projector_lr_scale 1.0
/usr/bin/python3 /content/embedding-kd/main.py --method geoode --train_data /content/embedding-kd/data/tra

In [ ]:
# 4. Chạy tuần tự; final-test record là resume boundary
from IPython.display import display

if EXECUTE:
    display(run_jobs(
        PROJECT_DIR, jobs, cuda_visible_devices=CUDA_VISIBLE_DEVICES,
        stop_on_error=STOP_ON_ERROR,
    ))
else:
    print("Dry run: đặt EXECUTE=True để chạy các job còn thiếu.")

[RUN 1/6] learned_t2s/seed_42 -> /content/embedding-kd/runs/analysis_interface_qwen3_0.6b_to_minilm_h384_v2/learned_t2s/seed_42

Configuration for GEOODE method:
  task_type                 : pair_cls
  max_length                : 256
  batch_size                : 128
  epochs                    : 5
  learning_rate             : 7e-05
  min_lr                    : 2e-06
  warmup_ratio              : 0.06
  w_task                    : 0.5
  alpha_dtw                 : 0.5
  w_cls                     : 1.0
  temperature               : 0.07
  student_model_name        : nreimers/MiniLMv2-L6-H384-distilled-from-BERT-Base
  teacher_model_name        : Qwen/Qwen3-Embedding-0.6B
  teacher_dtype             : bfloat16
  pooling_method            : last_token
  student_special_token     : ##
  teacher_special_token     : G
  train_data_path           : /content/embedding-kd/data/train_set/train_100k.csv
  cache_dir                 : /content/embedding-kd/runs/teacher_cache
  cache_batch_size  

In [ ]:
# 5. Appendix Table A1 — fixed geometry versus trainable interface
import numpy as np
import pandas as pd
import torch
from _analysis_common import load_teacher_cache, teacher_cache_path
from src import structural_audit as audit
from src.teacher_projection import retained_energy

results = collect_jobs(jobs)
results.to_csv(run_root / "interface_by_run.csv", index=False)
done = results.query("status == 'done'").copy()
if done.empty:
    print("No completed runs yet.")
else:
    expected = {"learned_t2s": 3, "learned_s2t": 3, "random_only": 9,
                "random_proc": 9, "pca_only": 3, "pca_proc": 3}
    counts = done.groupby("arm").size().to_dict()
    if REQUIRE_COMPLETE and counts != expected:
        raise RuntimeError(f"Incomplete A1 grid: got {counts}, expected {expected}")
    cache = teacher_cache_path(PROJECT_DIR, cache_dir, pair=pair, train_data=TRAIN_DATA)
    teacher, _ = load_teacher_cache(cache)
    rng = np.random.default_rng(0)
    rows = np.sort(rng.choice(len(teacher), min(GEOMETRY_ROWS, len(teacher)), replace=False))
    teacher = teacher[torch.as_tensor(rows)].float()
    geometry = []
    for kind, draws in (("pca", [0]), ("random", RANDOM_DRAWS)):
        for draw in draws:
            projection, mean = audit.fit_variant(teacher, kind, STUDENT_DIM, seed=draw)
            target = audit.apply_map(teacher, projection, mean=mean, subtract_mean=False)
            geometry.append({
                "geometry": kind, "draw": draw,
                "retained_energy": retained_energy(teacher, projection),
                "gram_rmse": audit.gram_rmse(target, teacher),
                "knn_overlap@10": audit.knn_overlap(target, teacher, k=10),
            })
    geometry = pd.DataFrame(geometry)
    geometry.to_csv(run_root / "interface_geometry_by_draw.csv", index=False)
    geom = geometry.groupby("geometry").agg(
        retained_energy=("retained_energy", "mean"),
        gram_rmse=("gram_rmse", "mean"),
        knn_overlap_at_10=("knn_overlap@10", "mean"),
    )
    score = done.groupby("arm").agg(
        final_mean=("avg_all", "mean"), final_sd=("avg_all", "std"), n=("avg_all", "count")
    )
    labels = {
        "learned_t2s": r"Learned $T\to S$", "learned_s2t": r"Learned $S\to T$",
        "random_only": "Random projection", "random_proc": "Random + epoch-wise Procrustes",
        "pca_only": "PCA", "pca_proc": "PCA + epoch-wise Procrustes",
    }
    geometry_for = {"random_only": "random", "random_proc": "random", "pca_only": "pca", "pca_proc": "pca"}
    rows = []
    for arm in labels:
        values = score.loc[arm]
        g = geom.loc[geometry_for[arm]] if arm in geometry_for else None
        rows.append({
            "Interface": labels[arm], "Fixed target": "No" if arm.startswith("learned") else "Yes",
            "Alignment": "Epoch-wise" if arm.endswith("proc") else "None",
            "Retained energy": np.nan if g is None else g.retained_energy,
            "Gram RMSE": np.nan if g is None else g.gram_rmse,
            "kNN overlap@10": np.nan if g is None else g.knn_overlap_at_10,
            "AVG": f"{100 * values.final_mean:.2f} ± {100 * values.final_sd:.2f}",
            "n": int(values.n),
        })
    table = pd.DataFrame(rows)
    table.to_csv(run_root / "table_A1_interface_ablation.csv", index=False)
    (run_root / "table_A1_interface_ablation.tex").write_text(
        table.drop(columns="n").to_latex(index=False, escape=False, na_rep="---"), encoding="utf-8"
    )
    display(table.style.format({"Retained energy": "{:.3f}", "Gram RMSE": "{:.4f}", "kNN overlap@10": "{:.3f}"}, na_rep="—"))